# SmartPantry AI — Role 1 Research: Shelf-Life Data Sourcing

**Hnin Aung · Role 1 (Research, Data Sources, Documentation) · ITAI 2277 · Fall 2026**

## Purpose

SmartPantry AI tells a user how many days a food has left and which item to eat first. Those
numbers have to come from somewhere trustworthy. This notebook documents exactly where they
come from, so every shelf-life value in the application can be traced back to a named United
States government source.

## How this supports the storage guidance component

The storage guidance feature (Role 3) answers "how many days does this food have left?" It
cannot answer that without a reference table of shelf-life values. This notebook produces that
reference table, with a source URL attached to every row.

**The division is deliberate:**

| This notebook (Role 1) | The application (Role 3) |
|---|---|
| Finds and verifies the data sources | Uses the data at runtime |
| Converts FoodKeeper values into days | Calculates days remaining from a user's date |
| Documents what the data does *not* cover | Returns `UNKNOWN` when no value exists |
| Exports a CSV with source links | Reads that CSV and ranks the pantry |

This notebook is **data sourcing and documentation only**. It contains no application logic and
is not imported by the running app.

## Original findings carried forward

1. The FoodSafety.gov chart our data was first built from does not cover produce or dairy.
2. FoodKeeper gives **no day count for milk**, so the app must return `UNKNOWN`.
3. The shelf-life clock **does not always start at the same point** — some foods are measured
   from purchase, others from opening.

## 0. Setup and requirements

### Required packages

| Package | Why it is needed |
|---|---|
| `pandas` | reading the Excel workbook and building the table |
| `numpy` | numeric handling and `NaN` for missing values |
| `xlrd` | pandas needs this to open the legacy `.xls` format FoodKeeper is published in |

### Install before running

Run this once in your terminal, **not** inside the notebook:

```
pip install pandas numpy xlrd
```

In Google Colab, run `!pip install xlrd` in a scratch cell before starting; `pandas` and
`numpy` are already present.

This notebook does **not** install packages while it runs. Installing during execution makes a
notebook non-reproducible and can silently change the environment, so the dependency step is
left to the person running it.

### Required data file

This notebook needs `FoodKeeper-Data.xls`. See section 2 for where to place it.

In [1]:
# Setup — imports and configuration only. No installs, no downloads.
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# --- paths -------------------------------------------------------------
# Paths are resolved relative to the *working directory the notebook is run
# from*, not to one person's folders, so the notebook behaves the same on any
# machine. Path.cwd() is the folder Jupyter was started in, which for a normal
# "open the notebook and run it" session is the folder the notebook sits in.
# If you launch Jupyter from somewhere else, set NOTEBOOK_DIR by hand to the
# folder holding this notebook.
NOTEBOOK_DIR = Path.cwd()
REPO_DATA    = NOTEBOOK_DIR.parent / "data" / "processed"  # when run from notebooks/ in the repo


def find_input(filename):
    """Look next to the notebook first, then in the repository data folder."""
    for candidate in (NOTEBOOK_DIR / filename, REPO_DATA / filename):
        if candidate.exists():
            return candidate
    return NOTEBOOK_DIR / filename          # not found; used for the error message


DATA_FILE    = find_input("FoodKeeper-Data.xls")                 # input, see section 2
OSU_FILENAME = "osu_extension_storage_rules_15.csv"              # input, see section 6c
OUTPUT_DIR   = NOTEBOOK_DIR / "role1_output"                     # output, see section 7
OUTPUT_FILE  = OUTPUT_DIR / "role1_foodkeeper_shelf_life.csv"

FOODKEEPER_URL = "https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls"
SOURCE_CITATION = (
    "USDA FSIS FoodKeeper Data (EN), "
    "https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls"
)

print("pandas", pd.__version__)
print("numpy ", np.__version__)
print("input files are looked for next to this notebook, then in data/processed/:")
print("   ", DATA_FILE.name, "->", "found" if DATA_FILE.exists() else "NOT FOUND")
_osu_probe = find_input(OSU_FILENAME)
print("   ", OSU_FILENAME, "->", "found" if _osu_probe.exists() else "NOT FOUND")
print("output folder:", OUTPUT_DIR.name + "/")

pandas 3.0.5
numpy  2.5.3
input files are looked for next to this notebook, then in data/processed/:
    FoodKeeper-Data.xls -> found
    osu_extension_storage_rules_15.csv -> found
output folder: role1_output/


## 1. The problem with our original source

The FoodSafety.gov Cold Food Storage Chart covers:

- meat, poultry, seafood
- eggs
- deli and packaged items
- leftovers

It does **not** list milk, yogurt, cheese, or any fresh fruit or vegetable. The FDA
*Refrigerator & Freezer Storage Chart* has the same gap. Both charts point readers to a
different USDA tool for everything else: **FoodKeeper**.

FoodKeeper is published by USDA Food Safety and Inspection Service, developed with Cornell
University and the Food Marketing Institute. It covers over 650 foods and the dataset is
public.

| Source | Covers | Used by |
|---|---|---|
| FoodSafety.gov Cold Food Storage Chart | meat, poultry, seafood, eggs, leftovers | storage guidance |
| FDA Refrigerator & Freezer Storage Chart | same gap as above | cross-check only |
| **USDA FoodKeeper** | **produce, dairy, packaged foods** | **this notebook** |

## 2. Load the FoodKeeper dataset

### Where the file goes

Place `FoodKeeper-Data.xls` **in the same folder as this notebook**. The setup cell prints the
exact path it expects, so if the file is missing you can see where to put it.

### If the USDA download is unavailable

The cell below tries the local file first, then falls back to downloading from USDA. The USDA
host is blocked on some networks and has returned HTTP 403 to automated requests during this
project, so the download may fail even though the file exists.

**If the download fails:**

1. Open <https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls> in a normal web browser
   and save the file.
2. If that URL is unavailable, search for "FoodKeeper data" on <https://catalog.data.gov> and
   download the EN `.xls` release.
3. Put the file next to this notebook using the exact name `FoodKeeper-Data.xls`.
4. Re-run this cell.

A failed download is not a failure of the research. It is a network restriction, and the
local-file path exists so the notebook stays reproducible either way.

In [2]:
def load_foodkeeper(path=DATA_FILE, url=FOODKEEPER_URL):
    """Open the FoodKeeper workbook.

    Tries the local file first so the notebook is reproducible offline.
    Falls back to the USDA download. Raises a clear, actionable error if neither works.
    """
    if path.exists():
        print("Using local file:", path.name)
        return pd.ExcelFile(path)

    print("Local file not found. Attempting download from USDA ...")
    try:
        raw = urllib.request.urlopen(url, timeout=60).read()
        path.write_bytes(raw)
        print("Downloaded", len(raw), "bytes to", path.name)
        return pd.ExcelFile(path)
    except Exception as exc:
        raise FileNotFoundError(
            "Could not load FoodKeeper data.\n"
            f"  Local file not found at: {path}\n"
            f"  Download also failed:    {exc}\n\n"
            "Fix: download the file in a web browser from\n"
            f"  {url}\n"
            f"and save it as '{path.name}' next to this notebook, then re-run this cell.\n"
            "See section 2 above for alternatives if that URL is unavailable."
        ) from exc


book = load_foodkeeper()
print("Sheets:", book.sheet_names)

Using local file: FoodKeeper-Data.xls
Sheets: ['Version', 'Category', 'Product', 'CookingTips', 'CookingMethods', 'Data Dictionary']


In [3]:
products   = book.parse("Product")
categories = book.parse("Category")

products["Name"] = products["Name"].astype(str)

print("Total foods in FoodKeeper:", len(products))
print("Total categories:", len(categories))
categories.head(10)

Total foods in FoodKeeper: 661
Total categories: 25


,ID,Category_Name,Subcategory_Name
0,1,Baby Food,NaN
1,2,Baked Goods,Bakery
2,3,Baked Goods,Baking and Cooking
3,4,Baked Goods,Refrigerated Dough
4,5,Beverages,NaN
5,6,"Condiments, Sauces & Canned Goods",NaN
6,7,Dairy Products & Eggs,NaN
7,8,Food Purchased Frozen,NaN
8,9,"Grains, Beans & Pasta",NaN
9,10,Meat,Fresh


## 3. One detail that matters for the application

FoodKeeper does not store a single "fridge days" number. It stores several, and they mean
different things:

| Column group | What the clock starts from |
|---|---|
| `DOP_Refrigerate_*` | the **date of purchase** |
| `Refrigerate_*` | simply refrigerated |
| `Refrigerate_After_Opening_*` | the day the package was **opened** |

This is not a formatting detail. Carrots last 2–3 weeks **from purchase**. Hard cheese lasts
3–4 weeks **from opening** — an unopened block keeps for months. If the application treats both
as "days since added", unopened cheese will be marked as spoiled long before it is.

Role 3 needs to know which basis each number uses, so the exported CSV carries a `basis`
column alongside every value.

In [4]:
TO_DAYS = {"Days": 1, "Weeks": 7, "Months": 30, "Years": 365}


def to_days(value, metric):
    """Convert a FoodKeeper value plus unit into days. Returns NaN if either is missing."""
    if pd.isna(value) or pd.isna(metric):
        return np.nan
    return float(value) * TO_DAYS.get(str(metric).strip(), np.nan)


def unopened_window(row):
    """Fridge life before the package is opened, with the basis it is measured from."""
    for lo, hi, met, basis in [
        ("DOP_Refrigerate_Min", "DOP_Refrigerate_Max", "DOP_Refrigerate_Metric", "from purchase"),
        ("Refrigerate_Min",     "Refrigerate_Max",     "Refrigerate_Metric",     "refrigerated"),
    ]:
        lo_d = to_days(row.get(lo), row.get(met))
        hi_d = to_days(row.get(hi), row.get(met))
        if not pd.isna(lo_d) or not pd.isna(hi_d):
            return lo_d, hi_d, basis
    return np.nan, np.nan, "no number given"


def opened_window(row):
    """Fridge life measured from the day the package was opened."""
    lo = to_days(row.get("Refrigerate_After_Opening_Min"), row.get("Refrigerate_After_Opening_Metric"))
    hi = to_days(row.get("Refrigerate_After_Opening_Max"), row.get("Refrigerate_After_Opening_Metric"))
    return lo, hi


print("Unit conversion table:", TO_DAYS)

Unit conversion table: {'Days': 1, 'Weeks': 7, 'Months': 30, 'Years': 365}


## 4. Pull the demo foods

At least one food from each food group, rather than a long list, so the demo proves the
application works across categories instead of on one lucky item. These are the foods selected,
with the FoodKeeper row each one matches.

In [5]:
category_label = dict(zip(
    categories["ID"],
    categories["Category_Name"].astype(str)
    + categories["Subcategory_Name"].fillna("").apply(lambda s: " - " + s if s else "")
))

# (FoodKeeper Name, text that must appear in Name_subtitle, or None for any)
TARGETS = [
    ("Milk",     "plain"),      # dairy
    ("Yogurt",   None),         # dairy
    ("Cheese",   "hard"),       # dairy
    ("Eggs",     "in shell"),   # eggs
    ("Lettuce",  "leaf"),       # vegetable
    ("Spinach",  None),         # vegetable
    ("Carrots",  None),         # vegetable
    ("Tomatoes", None),         # vegetable
    ("Apples",   None),         # fruit
    ("Bananas",  None),         # fruit
]


def find_row(name, subtitle):
    """Find a FoodKeeper row, recording how good the match was.

    FoodKeeper names do not always match the words a user would type.
    Returns (row, match_type) where match_type is 'exact', 'partial', or 'none'.
    """
    lowered = products["Name"].str.strip().str.lower()
    target = name.strip().lower()

    # 1. exact name match
    hits = products[lowered == target]
    match_type = "exact"

    # 2. fall back to a substring match, e.g. "Carrots" inside "Carrots, parsnips"
    if not len(hits):
        hits = products[lowered.str.contains(target, regex=False)]
        match_type = "partial" if len(hits) else "none"

    if not len(hits):
        return None, "none"

    if subtitle:
        sub = hits["Name_subtitle"].fillna("").str.lower()
        narrowed = hits[sub.str.contains(subtitle.lower())]
        if len(narrowed):
            hits = narrowed

    return hits.iloc[0], match_type


rows = []
for name, subtitle in TARGETS:
    row, match_type = find_row(name, subtitle)
    if row is None:
        rows.append({"searched_for": name, "food": "NOT IN FOODKEEPER",
                     "match": "none", "category": "n/a",
                     "unopened_min": np.nan, "unopened_max": np.nan,
                     "unopened_basis": "not found", "opened_min": np.nan,
                     "opened_max": np.nan,
                     "note": "no FoodKeeper row for this name"})
        continue
    lo, hi, basis = unopened_window(row)
    o_lo, o_hi = opened_window(row)
    matched_name = (f"{row['Name']}"
                    + (f" ({row['Name_subtitle']})" if pd.notna(row.get("Name_subtitle")) else ""))
    note = ""
    if basis == "no number given":
        note = "FoodKeeper gives no day count"
    elif match_type == "partial":
        note = f"name mismatch: searched '{name}', matched '{row['Name']}'"
    rows.append({
        "searched_for": name,
        "food": matched_name,
        "match": match_type,
        "category": category_label.get(row.get("Category_ID"), "unknown"),
        "unopened_min": lo,
        "unopened_max": hi,
        "unopened_basis": basis,
        "opened_min": o_lo,
        "opened_max": o_hi,
        "note": note,
    })

shelf_life = pd.DataFrame(rows)
shelf_life

,searched_for,food,match,category,unopened_min,unopened_max,unopened_basis,opened_min,opened_max,note
0,Milk,Milk (plain or flavored),exact,Dairy Products & Eggs,NaN,NaN,no number given,NaN,NaN,FoodKeeper gives no day count
1,Yogurt,Yogurt,exact,Dairy Products & Eggs,7.0,14.0,from purchase,NaN,NaN,
2,Cheese,"Cheese (hard such as cheddar, swiss, block par...",exact,Dairy Products & Eggs,180.0,180.0,from purchase,21.0,28.0,
3,Eggs,Eggs (in shell),exact,Dairy Products & Eggs,21.0,35.0,from purchase,NaN,NaN,
4,Lettuce,"Lettuce (leaf, spinach)",exact,Produce - Fresh Vegetables,3.0,7.0,from purchase,NaN,NaN,
5,Spinach,NOT IN FOODKEEPER,none,n/a,NaN,NaN,not found,NaN,NaN,no FoodKeeper row for this name
6,Carrots,"Carrots, parsnips",partial,Produce - Fresh Vegetables,14.0,21.0,from purchase,NaN,NaN,"name mismatch: searched 'Carrots', matched 'Ca..."
7,Tomatoes,Tomatoes,exact,Produce - Fresh Vegetables,NaN,NaN,no number given,NaN,NaN,FoodKeeper gives no day count
8,Apples,Apples,exact,Produce - Fresh Fruits,28.0,42.0,from purchase,NaN,NaN,
9,Bananas,Bananas,exact,Produce - Fresh Fruits,3.0,3.0,refrigerated,NaN,NaN,


## 5. Finding 1 — milk has no number

FoodKeeper gives no day count for milk. It says to follow the package use-by date.

This matters. Milk is the obvious food to demonstrate a fridge app with, and it is the one food
here our lookup cannot answer.

The decision: the application returns `UNKNOWN` for milk rather than inventing a date. A wrong
food safety answer is worse than no answer. The demo uses **yogurt** as its dairy example,
which does have a number.

In [6]:
no_number = shelf_life[shelf_life["unopened_basis"] == "no number given"]
print(f"Foods with no FoodKeeper day count: {len(no_number)}")
no_number[["food", "category", "unopened_basis", "note"]]

Foods with no FoodKeeper day count: 2


,food,category,unopened_basis,note
0,Milk (plain or flavored),Dairy Products & Eggs,no number given,FoodKeeper gives no day count
7,Tomatoes,Produce - Fresh Vegetables,no number given,FoodKeeper gives no day count


## 6. Finding 2 — the clock does not always start in the same place

Below, every food that has an "after opening" window. For hard cheese the two numbers are far
apart: months unopened, a few weeks once opened. Any food in this list needs the application to
know whether the item has been opened, not just when it was added.

In [7]:
both = shelf_life[shelf_life["opened_min"].notna()][
    ["food", "unopened_min", "unopened_max", "unopened_basis", "opened_min", "opened_max"]
]
print(f"Foods with a separate after-opening window: {len(both)}")
both

Foods with a separate after-opening window: 1


,food,unopened_min,unopened_max,unopened_basis,opened_min,opened_max
2,"Cheese (hard such as cheddar, swiss, block par...",180.0,180.0,from purchase,21.0,28.0


## 6b. Finding 3 — the names in FoodKeeper are not the names people use

Looking up each demo food by its everyday name exposed a third problem, and this one directly
affects whether the application can find a value at all.

FoodKeeper stores foods under its own naming scheme, which does not match what a user would
type or what image recognition returns:

| Searched for | What FoodKeeper actually calls it | Result with an exact-match lookup |
|---|---|---|
| Carrots | `Carrots, parsnips` | missed entirely |
| Spinach | bundled inside `Lettuce (leaf, spinach)` | no row of its own |

The lookup in this notebook now falls back to a substring match and records whether each match
was `exact` or `partial`, so a name mismatch is visible rather than silently returning nothing.

This is the same failure the team recorded from image recognition, where "Banana slices" did
not match "banana". It is one problem, not two: **there is no normalisation layer between the
name a food arrives under and the name FoodKeeper stores it under.**

Spinach is the harder case. It is not a naming mismatch that normalisation would fix — it has
no row of its own at all, and its values come from the combined lettuce entry. Any value the
application shows for spinach is really the lettuce figure, and that should be stated rather
than hidden.

In [8]:
match_report = shelf_life[["searched_for", "food", "match", "unopened_basis", "note"]]
print("Match quality across the demo foods:")
print(shelf_life["match"].value_counts().to_string())
match_report

Match quality across the demo foods:
match
exact      8
none       1
partial    1


,searched_for,food,match,unopened_basis,note
0,Milk,Milk (plain or flavored),exact,no number given,FoodKeeper gives no day count
1,Yogurt,Yogurt,exact,from purchase,
2,Cheese,"Cheese (hard such as cheddar, swiss, block par...",exact,from purchase,
3,Eggs,Eggs (in shell),exact,from purchase,
4,Lettuce,"Lettuce (leaf, spinach)",exact,from purchase,
5,Spinach,NOT IN FOODKEEPER,none,not found,no FoodKeeper row for this name
6,Carrots,"Carrots, parsnips",partial,from purchase,"name mismatch: searched 'Carrots', matched 'Ca..."
7,Tomatoes,Tomatoes,exact,no number given,FoodKeeper gives no day count
8,Apples,Apples,exact,from purchase,
9,Bananas,Bananas,exact,refrigerated,


## 6c. The second source — Oregon State University Extension

FoodKeeper does not list every food under its own name. The team therefore agreed a secondary
source: **Oregon State University Extension, *Storing Food for Safety and Quality* (PNW 612),
Table 3**, transcribed to CSV and supplied by Samira on 14 November 2026.

Source PDF: <https://extension.oregonstate.edu/sites/default/files/documents/pnw612.pdf>

### Where the file goes

Place `osu_extension_storage_rules_15.csv` next to this notebook, the same as the FoodKeeper
workbook. The cell below reports clearly if it is missing.

### What this source is used for

Only for foods FoodKeeper does not list individually. FoodKeeper remains primary, because it is
the dataset the application reads and because it states what each duration is measured from.

In [9]:
OSU_FILE = find_input(OSU_FILENAME)
OSU_CITATION = (
    "Oregon State University Extension, Storing Food for Safety and Quality (PNW 612), "
    "Table 3, https://extension.oregonstate.edu/sites/default/files/documents/pnw612.pdf"
)

if not OSU_FILE.exists():
    raise FileNotFoundError(
        f"Oregon State CSV '{OSU_FILENAME}' not found.\n"
        "Looked next to this notebook and in data/processed/.\n"
        "In the repository it lives at data/processed/" + OSU_FILENAME + "."
    )

osu = pd.read_csv(OSU_FILE)
print(f"Loaded {len(osu)} Oregon State rows from {OSU_FILE.name}")
osu[["food_type", "refrigerator_min_days", "refrigerator_max_days",
     "duration_basis", "source_notes"]]

Loaded 15 Oregon State rows from osu_extension_storage_rules_15.csv


,food_type,refrigerator_min_days,refrigerator_max_days,duration_basis,source_notes
0,Asparagus,3,4,NOT_SPECIFIED_IN_TABLE,NaN
1,"Beans, green or wax",7,7,NOT_SPECIFIED_IN_TABLE,NaN
2,Beets,7,10,NOT_SPECIFIED_IN_TABLE,NaN
3,Bok choy,2,3,NOT_SPECIFIED_IN_TABLE,NaN
4,"Broccoli, raab, rapini",3,5,NOT_SPECIFIED_IN_TABLE,Specific to broccoli raab/rapini; not generic ...
5,Brussels sprouts,3,5,NOT_SPECIFIED_IN_TABLE,NaN
6,Cauliflower,3,5,NOT_SPECIFIED_IN_TABLE,NaN
7,Cabbage,7,14,NOT_SPECIFIED_IN_TABLE,NaN
8,"Carrots, parsnips",21,21,NOT_SPECIFIED_IN_TABLE,Combined source entry; confirm item mapping
9,Celery,7,14,NOT_SPECIFIED_IN_TABLE,NaN


## 6d. Finding 4 — what the Oregon State file does and does not fix

Checking the file against the gaps it was meant to close gives a mixed result. The cell below
prints each check; the points worth recording are these.

**It genuinely fills two gaps.** Spinach is listed by name at 3 to 7 days, and green beans
appear as "Beans, green or wax" at 1 week. Neither has an individual FoodKeeper entry.

**It does not fill the pear gap.** This extraction contains fifteen vegetables and no fruit at
all, so pear is still only available through FoodKeeper's combined "Peaches, nectarines, plums,
pears, sapote" entry. The source publication may carry a separate fruit table that was not
transcribed.

**It repeats the carrot naming problem.** Oregon State also lists "Carrots, parsnips" as one
combined entry, exactly as FoodKeeper does, so that mismatch is unchanged.

**Its broccoli row is not generic broccoli.** The entry reads "Broccoli, raab, rapini" and the
file's own note says *"Specific to broccoli raab/rapini; not generic broccoli."* Broccoli raab
is a different vegetable. If a user typing "broccoli" is matched to this row, the application
shows a figure for a food they do not have.

**No row states what its duration is measured from.** Every row carries
`duration_basis = NOT_SPECIFIED_IN_TABLE`. FoodKeeper distinguishes "from date of purchase"
from "refrigerated"; Oregon State does not. An Oregon State value therefore cannot be anchored
to a purchase date with the same confidence as a FoodKeeper one, and that limitation should be
visible wherever such a value is shown.

Every row is also flagged `requires_condition_review = True` and `prototype_supported = False`
in the file as supplied.

In [10]:
osu_names = osu["food_type"].str.lower()

def osu_has(term):
    hits = osu[osu_names.str.contains(term, regex=False)]
    return hits.iloc[0] if len(hits) else None

print("Gaps this source was meant to fill")
print("-" * 60)
for label, term in [("spinach", "spinach"), ("green beans", "bean"), ("pear", "pear")]:
    row = osu_has(term)
    if row is None:
        print(f"  {label:12s} NOT PRESENT in this extraction")
    else:
        print(f"  {label:12s} '{row['food_type']}'  "
              f"{row['refrigerator_min_days']}-{row['refrigerator_max_days']} days")

print("\nRows whose name would not match what a user types")
print("-" * 60)
for _, r in osu.iterrows():
    if "," in str(r["food_type"]) or str(r["source_notes"]).strip() not in ("", "nan"):
        note = str(r["source_notes"]).strip()
        print(f"  {r['food_type']}")
        if note and note != "nan":
            print(f"      note: {note}")

print("\nMeasurement basis")
print("-" * 60)
print(" ", osu["duration_basis"].value_counts().to_string())
print(f"  rows flagged requires_condition_review: "
      f"{(osu['requires_condition_review'] == True).sum()} of {len(osu)}")
print(f"  rows flagged prototype_supported:       "
      f"{(osu['prototype_supported'] == True).sum()} of {len(osu)}")

Gaps this source was meant to fill
------------------------------------------------------------
  spinach      'Spinach'  3-7 days
  green beans  'Beans, green or wax'  7-7 days
  pear         NOT PRESENT in this extraction

Rows whose name would not match what a user types
------------------------------------------------------------
  Beans, green or wax
  Broccoli, raab, rapini
      note: Specific to broccoli raab/rapini; not generic broccoli
  Carrots, parsnips
      note: Combined source entry; confirm item mapping
  Corn on the cob
      note: For best flavor use corn immediately
  Mushrooms
      note: Do not wash before refrigerator storage
  Peppers, bell or chile

Measurement basis
------------------------------------------------------------
  duration_basis
NOT_SPECIFIED_IN_TABLE    15
  rows flagged requires_condition_review: 15 of 15
  rows flagged prototype_supported:       0 of 15


## 7. Export for Role 3

### Output location

The CSV is written to a dedicated folder next to this notebook:

```
role1_output/role1_foodkeeper_shelf_life.csv
```

### This file is research output, not the dataset the application reads

To be explicit, because the two are easy to confuse:

| | This notebook's CSV | The application's dataset |
|---|---|---|
| Path | `role1_output/role1_foodkeeper_shelf_life.csv` | `data/raw/foodkeeper_produce_dairy_shelf_life.csv` |
| Purpose | research evidence for the ten demo foods | the data the storage module actually reads |
| Written by | re-running this notebook | a deliberate, reviewed commit |
| Contains | USDA FoodKeeper values only | the project's maintained dataset |

**They are not the same file, and this notebook does not update the application's dataset.**
Re-running a notebook should never silently overwrite data the application depends on.

Promoting anything from here into the project is a separate, deliberate step: review the file,
then copy the relevant values into `data/raw/` as its own commit.

### Both sources appear in this export

Each row names the source its figure came from, in two columns:

| Column | Meaning |
|---|---|
| `source_name` | short name: `USDA FSIS FoodKeeper` or `Oregon State University Extension` |
| `source` | the full citation, including the URL |

FoodKeeper is used wherever it lists the food individually. Oregon State is used only where it
does not. In the current run that means nine rows from FoodKeeper and one, spinach, from
Oregon State.

Rows filled from Oregon State also carry `unopened_basis = "not stated by source"`, because
Oregon State's table does not say whether its durations run from purchase or from
refrigeration. That is recorded rather than assumed.

In [11]:
OUTPUT_DIR.mkdir(exist_ok=True)

export = shelf_life.copy()

# Every row starts as FoodKeeper-sourced, then Oregon State fills the gaps.
export["source_name"] = "USDA FSIS FoodKeeper"
export["source"]      = SOURCE_CITATION

# Map the foods we searched for to the Oregon State row that covers them.
OSU_FALLBACK = {
    "Spinach":     "spinach",
    "Green beans": "bean",
}

filled = []
for idx, row in export.iterrows():
    needs_fallback = row["match"] in ("none", "partial") or row["unopened_basis"] in (
        "not found", "no number given"
    )
    term = OSU_FALLBACK.get(row["searched_for"])
    if not (needs_fallback and term):
        continue

    hits = osu[osu["food_type"].str.lower().str.contains(term, regex=False)]
    if not len(hits):
        continue

    o = hits.iloc[0]
    export.at[idx, "food"]           = o["food_type"]
    export.at[idx, "match"]          = "oregon state"
    export.at[idx, "unopened_min"]   = float(o["refrigerator_min_days"])
    export.at[idx, "unopened_max"]   = float(o["refrigerator_max_days"])
    export.at[idx, "unopened_basis"] = "not stated by source"
    export.at[idx, "source_name"]    = "Oregon State University Extension"
    export.at[idx, "source"]         = OSU_CITATION
    export.at[idx, "note"]           = (
        "FoodKeeper has no individual entry; value from Oregon State. "
        "Source does not state what the duration is measured from."
    )
    filled.append(f"{row['searched_for']} -> {o['food_type']}")

export.to_csv(OUTPUT_FILE, index=False)

print(f"Wrote {OUTPUT_DIR.name}/{OUTPUT_FILE.name}")
print(f"Rows: {len(export)}")
print("\nRows filled from Oregon State:")
for f in filled:
    print("  ", f)
print("\nRows by source:")
print(export["source_name"].value_counts().to_string())

export[["searched_for", "food", "match", "unopened_min", "unopened_max",
        "unopened_basis", "source_name"]]

Wrote role1_output/role1_foodkeeper_shelf_life.csv
Rows: 10

Rows filled from Oregon State:
   Spinach -> Spinach

Rows by source:
source_name
USDA FSIS FoodKeeper                 9
Oregon State University Extension    1


,searched_for,food,match,unopened_min,unopened_max,unopened_basis,source_name
0,Milk,Milk (plain or flavored),exact,NaN,NaN,no number given,USDA FSIS FoodKeeper
1,Yogurt,Yogurt,exact,7.0,14.0,from purchase,USDA FSIS FoodKeeper
2,Cheese,"Cheese (hard such as cheddar, swiss, block par...",exact,180.0,180.0,from purchase,USDA FSIS FoodKeeper
3,Eggs,Eggs (in shell),exact,21.0,35.0,from purchase,USDA FSIS FoodKeeper
4,Lettuce,"Lettuce (leaf, spinach)",exact,3.0,7.0,from purchase,USDA FSIS FoodKeeper
5,Spinach,Spinach,oregon state,3.0,7.0,not stated by source,Oregon State University Extension
6,Carrots,"Carrots, parsnips",partial,14.0,21.0,from purchase,USDA FSIS FoodKeeper
7,Tomatoes,Tomatoes,exact,NaN,NaN,no number given,USDA FSIS FoodKeeper
8,Apples,Apples,exact,28.0,42.0,from purchase,USDA FSIS FoodKeeper
9,Bananas,Bananas,exact,3.0,3.0,refrigerated,USDA FSIS FoodKeeper


## 8. Testing and validation

Six checks run below. Each one prints its own result, and the summary table at the end is
built from what actually happened in this run — the values are not typed in by hand.

| # | Test performed | Expected result |
|---|---|---|
| 1 | Dataset loading | The Product and Category sheets load with more than 500 food rows |
| 2 | Unit conversion | `Weeks` converts at 7 days, `Months` at 30, missing input returns `NaN` |
| 3 | Missing shelf-life values | Milk resolves to `no number given` rather than a numeric date |
| 4 | CSV export | The output file exists, row count matches, and `source` is present on every row |
| 5 | Food name matching | Every searched food is reported, with exact or partial match recorded, never silently dropped |
| 6 | Two-source export | Every row names its source, and both FoodKeeper and Oregon State appear |

In [12]:
results = []


def check(test_id, description, expected, actual, passed):
    results.append({
        "#": test_id,
        "Test performed": description,
        "Expected result": expected,
        "Actual result": actual,
        "Status": "PASS" if passed else "FAIL",
    })
    print(f"[{'PASS' if passed else 'FAIL'}] {test_id}. {description}")
    print(f"       expected: {expected}")
    print(f"       actual:   {actual}\n")


# --- Test 1: dataset loading ------------------------------------------
n_products = len(products)
n_categories = len(categories)
check(
    1, "Dataset loading",
    "Product and Category sheets load, over 500 food rows",
    f"Product sheet {n_products} rows, Category sheet {n_categories} rows",
    n_products > 500 and n_categories > 0,
)

# --- Test 2: unit conversion ------------------------------------------
weeks_2   = to_days(2, "Weeks")      # expect 14
months_1  = to_days(1, "Months")     # expect 30
missing   = to_days(np.nan, "Days")  # expect NaN
conv_ok = (weeks_2 == 14) and (months_1 == 30) and pd.isna(missing)
check(
    2, "Unit conversion",
    "2 Weeks = 14 days, 1 Month = 30 days, missing value = NaN",
    f"2 Weeks = {weeks_2}, 1 Month = {months_1}, missing = {missing}",
    conv_ok,
)

# --- Test 3: missing shelf-life values --------------------------------
milk_rows  = shelf_life[shelf_life["food"].str.lower().str.startswith("milk")]
milk_basis = milk_rows["unopened_basis"].iloc[0] if len(milk_rows) else "milk row not found"
milk_min   = milk_rows["unopened_min"].iloc[0] if len(milk_rows) else None
milk_ok = (milk_basis == "no number given") and pd.isna(milk_min)
check(
    3, "Missing shelf-life values",
    "Milk returns 'no number given' and no numeric day count",
    f"basis = '{milk_basis}', unopened_min = {milk_min}",
    milk_ok,
)

# --- Test 4: CSV export -----------------------------------------------
exists    = OUTPUT_FILE.exists()
reloaded  = pd.read_csv(OUTPUT_FILE) if exists else pd.DataFrame()
rows_match = len(reloaded) == len(shelf_life)
has_source = "source" in reloaded.columns and reloaded["source"].notna().all()
check(
    4, "CSV export",
    "File exists, row count matches, source present on every row",
    f"exists = {exists}, rows = {len(reloaded)} of {len(shelf_life)}, "
    f"source on all rows = {has_source}",
    exists and rows_match and has_source,
)

# --- Test 5: name matching ---------------------------------------------
partial = shelf_life[shelf_life["match"] == "partial"]
not_found = shelf_life[shelf_life["match"] == "none"]
check(
    5, "Food name matching",
    "Every demo food either matches FoodKeeper or is reported as unmatched, never silently dropped",
    f"exact = {(shelf_life['match'] == 'exact').sum()}, "
    f"partial = {len(partial)}, not found = {len(not_found)}, "
    f"total rows = {len(shelf_life)} of {len(TARGETS)} searched",
    len(shelf_life) == len(TARGETS),
)

# --- Test 6: both sources present and cited ------------------------------
reloaded6  = pd.read_csv(OUTPUT_FILE)
has_cols   = {"source_name", "source"}.issubset(reloaded6.columns)
all_named  = reloaded6["source_name"].notna().all() and reloaded6["source"].notna().all()
n_sources  = reloaded6["source_name"].nunique()
counts     = reloaded6["source_name"].value_counts().to_dict()
check(
    6, "Two-source export",
    "Every row names its source, and both FoodKeeper and Oregon State appear",
    f"columns present = {has_cols}, every row cited = {all_named}, "
    f"distinct sources = {n_sources}, breakdown = {counts}",
    has_cols and all_named and n_sources == 2,
)

summary = pd.DataFrame(results)
passed = (summary["Status"] == "PASS").sum()
print(f"=== {passed} of {len(summary)} checks passed ===")
summary

[PASS] 1. Dataset loading
       expected: Product and Category sheets load, over 500 food rows
       actual:   Product sheet 661 rows, Category sheet 25 rows

[PASS] 2. Unit conversion
       expected: 2 Weeks = 14 days, 1 Month = 30 days, missing value = NaN
       actual:   2 Weeks = 14.0, 1 Month = 30.0, missing = nan

[PASS] 3. Missing shelf-life values
       expected: Milk returns 'no number given' and no numeric day count
       actual:   basis = 'no number given', unopened_min = nan



[PASS] 4. CSV export
       expected: File exists, row count matches, source present on every row
       actual:   exists = True, rows = 10 of 10, source on all rows = True

[PASS] 5. Food name matching
       expected: Every demo food either matches FoodKeeper or is reported as unmatched, never silently dropped
       actual:   exact = 8, partial = 1, not found = 1, total rows = 10 of 10 searched

[PASS] 6. Two-source export
       expected: Every row names its source, and both FoodKeeper and Oregon State appear
       actual:   columns present = True, every row cited = True, distinct sources = 2, breakdown = {'USDA FSIS FoodKeeper': 9, 'Oregon State University Extension': 1}

=== 6 of 6 checks passed ===


,#,Test performed,Expected result,Actual result,Status
0,1,Dataset loading,"Product and Category sheets load, over 500 foo...","Product sheet 661 rows, Category sheet 25 rows",PASS
1,2,Unit conversion,"2 Weeks = 14 days, 1 Month = 30 days, missing ...","2 Weeks = 14.0, 1 Month = 30.0, missing = nan",PASS
2,3,Missing shelf-life values,Milk returns 'no number given' and no numeric ...,"basis = 'no number given', unopened_min = nan",PASS
3,4,CSV export,"File exists, row count matches, source present...","exists = True, rows = 10 of 10, source on all ...",PASS
4,5,Food name matching,Every demo food either matches FoodKeeper or i...,"exact = 8, partial = 1, not found = 1, total r...",PASS
5,6,Two-source export,"Every row names its source, and both FoodKeepe...","columns present = True, every row cited = True...",PASS


## 9. Phase 4 integration

### What this notebook provides to Role 3

A CSV of shelf-life values in days, with the measurement basis and a source citation on every
row. The storage guidance module reads that table to answer "how many days does this food have
left?"

### What is currently implemented in SmartPantry

Confirmed against the deployed application on 8 November 2026, and with Samira (integration) on
the same day.

| Capability | Status |
|---|---|
| FoodKeeper values available for produce and dairy | Implemented — integrated during Phase 3 |
| `UNKNOWN` returned where no shelf-life value exists | Implemented in the storage module |
| Storage guidance connected to the pantry | **Not yet connected** |
| Purchase date captured when adding food | **Not yet built** |
| Pantry sorted by what needs eating first | **Not yet connected** |

The pantry currently records name, quantity and unit only. There is no date field and no
storage-location field, so the application cannot yet calculate days remaining from a pantry
entry. This is expected at this point in Phase 4: the storage data is still being assembled and
will be connected once it is complete.

This table describes what the application does today, not what is planned. It was checked
directly rather than assumed.

### What still needs development

| Gap | Why it matters |
|---|---|
| No purchase or opening date is captured | Without a start date no countdown can be calculated at all |
| Opened versus unopened state is not tracked | Hard cheese is treated as if the opening clock already started |
| Food name normalisation | "Banana slices" does not match "banana" in the lookup table |
| Foods measured from refrigeration rather than purchase | Those cannot use a purchase date, so they return `UNKNOWN` |
| No day count published for fresh milk | The application must ask the user for the package date |

### Why purchase dates and opening dates matter

FoodKeeper measures different foods from different starting points. Carrots are measured from
the **date of purchase**. Hard cheese is measured from the **day the package was opened**, and
an unopened block keeps for months.

If the application assumes a single "date added" for both, unopened cheese is flagged as
spoiled weeks early, and a food measured from refrigeration cannot be calculated at all. The
`unopened_basis` column exists so the storage module can tell these cases apart.

### Why missing information returns UNKNOWN

FoodKeeper gives no day count for milk; it defers to the printed use-by date. Checking further,
the University of Delaware Cooperative Extension refrigerator guide has no entry for fresh milk
either, and Oregon State University Extension does not publish one.

This is not a weakness in the sources chosen. Fresh pasteurised milk has no fixed shelf life:
it depends on the pasteurisation method, the actual temperature of the refrigerator, and how
often the container is opened. Every authoritative source therefore defers to the printed date
rather than publishing a number.

The application returns `UNKNOWN` and asks the user for the package date. That is not a
workaround. It is what the published guidance itself instructs.

## 10. Summary

- The FoodSafety.gov chart our shelf-life data was first built from does not cover produce or
  dairy. Neither does the FDA chart. Both point to USDA FoodKeeper instead.
- FoodKeeper supplied values for every food group the demo needs, pulled here directly from the
  USDA file so the numbers are reproducible rather than retyped.
- **Milk has no day count** in FoodKeeper, and no other authoritative source publishes one
  either. The application returns `UNKNOWN` and asks for the package date.
- **The countdown starts at different points** depending on the food — purchase date for most,
  opening date for others. The exported `basis` column carries that distinction through.
- **FoodKeeper's names are not the names people use.** Carrots are listed as "Carrots,
  parsnips", and spinach has no entry of its own at all.
- Every exported row carries its source URL, so any number in the application can be traced.

## Documented data sources

Two sources are used, in a defined order, so every value can still name where it came from.

| Source | Role | Used for | In this notebook's export? |
|---|---|---|---|
| USDA FSIS FoodKeeper | Primary | All foods with a named FoodKeeper entry | Yes |
| Oregon State University Extension, *Storing Food for Safety and Quality* (PNW 612) | Secondary | Foods FoodKeeper does not list under their own name | Yes |

Oregon State was added deliberately rather than as a convenience. FoodKeeper lumps several
common foods into combined entries, and Oregon State lists them individually:

| Food | FoodKeeper | Oregon State |
|---|---|---|
| Spinach | no entry; nearest is "Lettuce (leaf, spinach)" | listed by name, 3–7 days |
| Pear | inside "Peaches, nectarines, plums, pears, sapote" | listed by name, 3–4 days |
| Green beans | inside "Beans and peas" | listed by name, 1 week |

**Current state:** both sources are now in the export. Every row carries `source_name` and a
full `source` citation, so each figure names where it came from. FoodKeeper covers nine of the
ten demo foods; spinach comes from Oregon State, because FoodKeeper has no spinach entry.

**What Oregon State does not fix:** it has no fruit in this extraction, so pear is still only
available through FoodKeeper's combined entry. It lists "Carrots, parsnips" combined, exactly
as FoodKeeper does. Its broccoli row is "Broccoli, raab, rapini", a different vegetable from
ordinary broccoli. And no row states what its duration is measured from. Section 6d records
these in full.